# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.5 | NumPy 2.3.3 | pandas 3.0.5
Wrote 5 data files to /Users/yassa/Desktop/Fall 2026/DS 217/VS Code/ds217-26f-06/data
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
def profile(name, table):
    """Return the four Task 1.2 values for one data file."""
    return {
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
        "exact_duplicate_rows": table.duplicated().sum(),
    }


results = []
for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))

table_profile = pd.DataFrame(results)
display(table_profile)
table_profile.to_csv(PROFILE_PATH, index=False)# TODO: write a function, such as profile(name, table), that returns the four values in README


,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

Grain and keys of the cleaned tables 

`patients_clean`: I have one row per registered patient, so `patient_id` is the key. There were 45 exact duplicate rows in the raw file that would have caused problems with this.
`sites_current`: I kept one row per site, using only the current records, so `site_code` is the key. In the original `sites.csv`, N01 and S02 each appeared twice, once as a current record and once as a retired one. W03 does not have any patients, and E04 appears in the registry but is not in the site list.
`visits_clean`: After stacking the north and south files and keeping the later recheck, I ended up with one row per patient per visit. The key is `patient_id` + `visit`. In the raw files, there were 70 exact duplicate rows (35 in each file) and 120 rechecks where the same patient and visit were measured twice within a few minutes. I had to map the visit labels first (`wk12` = `week12`) for these rechecks to show up as the same visit.
`a1c_wide`: I have one row per patient, with `patient_id` as the key. There were no repeated patient IDs in this table.
`a1c_long`: After melting the data and dropping the missing results, I have one row per patient per visit with an actual result. The key is `patient_id` + `visit`.

Expected cardinality of the Task 4.4 joins:
Visits to patients on `patient_id`: I expect this to be a many-to-one join since a patient can have up to two visits, but each visit should belong to one patient. Some visits might not have a matching patient in the registry, so I would still keep those rows and leave the patient information missing.  Patients to sites on `site_code`: I expect this to also be many-to-one since multiple patients can belong to the same site. E04 is an exception since it appears in the registry but does not have a matching site record, so those patients will not have a site name. Visits to A1c results on `patient_id` + `visit`: I expect this to be one-to-one since there should be at most one A1c result for each patient and visit. I used a left join so that visits without an A1c result would still be kept.

### 2.2 Stack the two visit files

In [12]:
north = raw["visits_north"].copy(deep=True)
south = raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"})
north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"

visits_raw = pd.concat([north, south], ignore_index=True)
print("shape:", visits_raw.shape)
print(visits_raw.isna().all())

shape: (57371, 7)
patient_id     False
visit          False
visit_date     False
visit_time     False
sbp_mmhg       False
weight_kg      False
source_file    False
dtype: bool


### 2.3 Audit the raw tables

In [13]:
patients = raw["patients"]
sites = raw["sites"]

# patients
gender = patients["gender"]
not_a_code = (gender.str.strip() != "") & ~gender.isin(["F", "M", "X"])
empty_gender = gender.str.strip() == ""
sentinel_year = patients["birth_year"] == "9999"
site = patients["site_code"]
site_normalized = site.str.upper().str.replace("-", "").str.replace(" ", "")
site_needs_fix = site_normalized != site

# sites
repeated_site = sites.duplicated(subset=["site_code"], keep=False)

# visits
visit_map = {"baseline": "baseline", "bl": "baseline", "week12": "week12", "wk12": "week12", "week 12": "week12"}
bad_label = ~visits_raw["visit"].isin(["baseline", "week12"])
sbp_text = visits_raw["sbp_mmhg"]
sbp_number = pd.to_numeric(sbp_text, errors="coerce")
sbp_sentinel = sbp_text == "999"
sbp_outside = (sbp_number != 999) & ((sbp_number < 70) | (sbp_number > 250))
date_text = visits_raw["visit_date"]
exact_date = date_text.str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")
parsed_date = pd.to_datetime(date_text.where(exact_date), format="%Y-%m-%d", errors="coerce")
date_failure = parsed_date.isna()
pairs = pd.DataFrame({
    "patient_id": visits_raw["patient_id"],
    "visit": visits_raw["visit"].str.strip().str.lower().map(visit_map),
})
repeated_pair = pairs.duplicated(subset=["patient_id", "visit"], keep=False)

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs",
    ],
    "count": [
        not_a_code.sum(),
        empty_gender.sum(),
        sentinel_year.sum(),
        site_needs_fix.sum(),
        repeated_site.sum(),
        bad_label.sum(),
        sbp_sentinel.sum(),
        sbp_outside.sum(),
        date_failure.sum(),
        repeated_pair.sum(),
    ],
})
display(issue_audit)
issue_audit.to_csv(AUDIT_PATH, index=False)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

,issue,count
0,patients: gender values that are not a code,6968
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,380


## Task 3: Clean

### 3.1 Clean the patients

In [6]:
display(raw["patients"]["gender"].value_counts())

patients_clean = raw["patients"].copy(deep=True)
patients_clean = patients_clean.drop_duplicates()

gender_map = {
    "f": "F", "female": "F", "woman": "F",
    "m": "M", "male": "M", "man": "M",
    "x": "X", "nb": "X", "nonbinary": "X", "non-binary": "X",
}
patients_clean["gender"] = (
    patients_clean["gender"].str.strip().str.lower().map(gender_map).astype("category")
)

birth_year = pd.to_numeric(patients_clean["birth_year"], errors="coerce")
patients_clean["birth_year"] = birth_year.where(birth_year != 9999).astype("Int64")

patients_clean["site_code"] = (
    patients_clean["site_code"].str.upper().str.replace("-", "").str.replace(" ", "")
)

patients_clean = patients_clean.reset_index(drop=True)
patients_clean.head()

gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]

,patient_id,gender,birth_year,site_code
0,P00001,F,1966,S02
1,P00002,F,<NA>,N01
2,P00003,F,1970,N01
3,P00004,X,<NA>,N01
4,P00005,M,1947,N01


In [7]:
print(patients_clean["gender"].cat.categories)

patients_summary = pd.DataFrame({
    "metric": [
        "patients",
        "birth_year missing",
        "birth_year min",
        "birth_year max",
        "site_code values",
        "patients at N01",
        "patients at S02",
        "patients at E04",
    ],
    "value": [
        len(patients_clean),
        patients_clean["birth_year"].isna().sum(),
        patients_clean["birth_year"].min(),
        patients_clean["birth_year"].max(),
        patients_clean["site_code"].nunique(),
        (patients_clean["site_code"] == "N01").sum(),
        (patients_clean["site_code"] == "S02").sum(),
        (patients_clean["site_code"] == "E04").sum(),
    ],
})
display(patients_summary)
patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
display(gender_counts)
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

Index(['F', 'M', 'X'], dtype='str')


,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


,gender,count,code
0,F,14413,0
1,M,13955,1
2,NaN,863,-1
3,X,769,2


### 3.2 Clean the visits

In [8]:
display(visits_raw["visit"].value_counts())

visits_clean = visits_raw.copy(deep=True)
visits_clean = visits_clean.drop_duplicates()

visit_map = {"baseline": "baseline", "bl": "baseline", "week12": "week12", "wk12": "week12", "week 12": "week12"}
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(visit_map)

sbp = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = sbp.where(sbp.between(70, 250)).astype("Int64")

visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce")

exact = visits_clean["visit_date"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")
visits_clean["visit_date"] = pd.to_datetime(
    visits_clean["visit_date"].where(exact), format="%Y-%m-%d", errors="coerce"
)

print("visits after removing exact repeats:", len(visits_clean))

visit
baseline     28533
week12       26529
BASELINE       256
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        195
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]

visits after removing exact repeats: 57301


In [9]:
rechecks = visits_clean[visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)]
display(rechecks.sort_values(["patient_id", "visit", "visit_time"]).head(10))
print("rows in repeated pairs:", len(rechecks))

visits_clean = visits_clean.sort_values("visit_time")
visits_clean = visits_clean.drop_duplicates(subset=["patient_id", "visit"], keep="last")
print("repeated pairs left:", visits_clean.duplicated(subset=["patient_id", "visit"]).sum())

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
85,P00088,baseline,2026-03-10,09:50,152,68.2,visits_north.csv
26491,P00183,week12,2026-05-25,14:15,134,78.9,visits_south.csv
26490,P00183,week12,2026-05-25,14:35,133,78.9,visits_south.csv
26684,P00377,baseline,2026-01-30,09:05,147,101.0,visits_south.csv
26685,P00377,baseline,2026-01-30,09:15,139,101.0,visits_south.csv


rows in repeated pairs: 240
repeated pairs left: 0


In [10]:
visits_clean = visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)
visits_clean = visits_clean[["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]]
display(visits_clean["visit"].value_counts(dropna=False))

visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv",
    ],
    "value": [
        len(visits_clean),
        (visits_clean["visit"] == "baseline").sum(),
        (visits_clean["visit"] == "week12").sum(),
        visits_clean["visit_date"].isna().sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        visits_clean["sbp_mmhg"].isna().sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        visits_clean["weight_kg"].isna().sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        (visits_clean["source_file"] == "visits_north.csv").sum(),
        (visits_clean["source_file"] == "visits_south.csv").sum(),
    ],
})
display(visits_summary)
visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)

visit
baseline    29599
week12      27582
Name: count, dtype: int64

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [11]:
decision_table = pd.DataFrame({
    "table": ["patients", "patients", "patients", "patients", "visits", "visits", "visits", "visits", "sites"],
    "issue": [
        "exact duplicate rows",
        "gender spellings that are not a code",
        "site codes needing normalization",
        "birth_year sentinel 9999",
        "weight column named wt_kg in visits_south.csv",
        "sbp sentinel 999 and readings outside 70 through 250",
        "impossible and loosely written visit dates",
        "repeated patient_id and visit pairs",
        "repeated site_code rows",
    ],
    "decision": [
        "I used drop_duplicates() to remove the exact duplicate rows, keeping the first copy. There did not seem to be a reason to keep identical rows more than once.",
        "I stripped the spaces and made the gender values lowercase before mapping the different spellings to F, M, or X using a dictionary. Anything that did not match was left missing, along with the blank values, and I stored the column as a category.",
        "I made the site codes uppercase and removed the hyphens and spaces so they would match the other files more easily. I left E04 as it was since that is how it was recorded in the registry.",
        "I changed the 9999 values to missing and stored birth_year as Int64, since 9999 seemed to be used as a placeholder for years that were not recorded.",
        "I renamed wt_kg to weight_kg in a copy of the south file before combining the two visit files. This seemed like the simplest way to keep the weights in one column.",
        "I changed the 999 values and readings below 70 or above 250 to missing, but kept the visits themselves. I figured it was better to leave the other information intact rather than remove the entire row. Values of exactly 70 and 250 were kept since they are still within the range.",
        "I kept visit_date only when it was written exactly as YYYY-MM-DD and represented a real calendar date. The other values were changed to missing, but I kept the rest of the visit since that information could still be useful.",
        "I sorted by visit_time and used drop_duplicates() with keep='last' to keep the later reading for each patient_id and visit pair. These seemed to be rechecks from the same visit, so keeping the later one made the most sense to me.",
        "I kept the current record for each site_code in sites_current and left out the retired records. This seemed like the best way to avoid having multiple records for the same site when joining the data.",
    ],
    "reason": [
        "There were 45 rows that were exact copies of another registry row. Keeping these would probably end up counting the same patients more than once, so I removed them. This left 30000 patients, with one row per patient_id.",
        "There were 6968 gender values that were not written as F, M, or X, mostly things like woman, Male, or nb, along with 486 blank values. I was not really sure how to handle these without making assumptions about what they meant, so I left them missing rather than creating another code. This left 863 patients with missing gender values.",
        "I found 1811 site codes written in different ways, such as n-01, N 01, or s02. These could cause issues when joining with sites.csv because the same site might end up being treated as multiple different sites. E04 was kept since it was the code recorded for 555 patients, and I did not want to change it without a reason.",
        "There were 371 patients with birth_year recorded as 9999, which seemed to mean that the year was not recorded. Leaving these values as numbers would make the maximum birth year 9999, which would obviously cause problems if I tried to calculate age.",
        "The south office used wt_kg instead of weight_kg for the weight column. Without renaming it, stacking the files would probably leave me with two separate weight columns, so I changed the name before combining them. This gave me one weight_kg column across all 57371 visit rows.",
        "There were 341 readings recorded as 999, which seemed to be the code for a measurement that was not taken, and another 116 readings outside the 70 to 250 range. These did not seem like valid blood pressure readings, but the rest of the visit was still usable. I only made the SBP values missing and kept exactly 70 and 250 since the range is inclusive.",
        "I found 169 dates that were either impossible or written in a different format, such as 2026-2-10 or 2026/01/22. I was not comfortable guessing what the intended date was since that could put the visit on the wrong day. The rest of the visit information was still usable, so I only changed the date to missing.",
        "After removing the exact duplicates, there were still 120 visits with two readings taken minutes apart. These looked like rechecks from the same visit, so I kept the later reading to get one row per patient_id and visit. This should also make the pivot and one-to-one A1c join work properly.",
        "N01 and S02 each had a current and a retired record in sites.csv. Joining all of these would probably duplicate the visits for those sites, so I kept only the current records. This gave me one site name per site_code and removed the 2 retired rows.",
    ],
})
display(decision_table)
decision_table.to_csv(DECISION_LOG_PATH, index=False)

,table,issue,decision,reason
0,patients,exact duplicate rows,I used drop_duplicates() to remove the exact d...,There were 45 rows that were exact copies of a...
1,patients,gender spellings that are not a code,I stripped the spaces and made the gender valu...,There were 6968 gender values that were not wr...
2,patients,site codes needing normalization,I made the site codes uppercase and removed th...,I found 1811 site codes written in different w...
3,patients,birth_year sentinel 9999,I changed the 9999 values to missing and store...,There were 371 patients with birth_year record...
4,visits,weight column named wt_kg in visits_south.csv,I renamed wt_kg to weight_kg in a copy of the ...,The south office used wt_kg instead of weight_...
5,visits,sbp sentinel 999 and readings outside 70 throu...,I changed the 999 values and readings below 70...,"There were 341 readings recorded as 999, which..."
6,visits,impossible and loosely written visit dates,I kept visit_date only when it was written exa...,I found 169 dates that were either impossible ...
7,visits,repeated patient_id and visit pairs,I sorted by visit_time and used drop_duplicate...,"After removing the exact duplicates, there wer..."
8,sites,repeated site_code rows,I kept the current record for each site_code i...,N01 and S02 each had a current and a retired r...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [ ]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = None

### 4.2 Check the keys before joining

In [ ]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [ ]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [ ]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [ ]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [ ]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [ ]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [ ]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [ ]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.